## Text input

https://platform.openai.com/docs/models

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import requests

# Check if Ollama is running
ollama_url = "http://localhost:11434/api/tags"
try:
    response = requests.get(ollama_url, timeout=2)
    if response.status_code == 200:
        print("✓ Ollama is running")
        models = response.json().get('models', [])
        if models:
            print(f"✓ Found {len(models)} model(s):")
            for model in models:
                print(f"  - {model['name']}")
        else:
            print("⚠ No models found. Run: ollama pull llama3.2")
    else:
        raise Exception(f"Ollama returned status {response.status_code}")
except requests.exceptions.ConnectionError:
    raise RuntimeError(
        "❌ Ollama is not running!\n"
        "Please start Ollama with: ollama serve\n"
        "Or install from: https://ollama.ai"
    )
except Exception as e:
    raise RuntimeError(f"❌ Error connecting to Ollama: {e}")

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

# Create the Ollama model upfront
llm = ChatOllama(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.7,
)

# Pass the model to create_agent
agent = create_agent(
    model=llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

In [1]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [16]:
print(uploader.value)
print(type(uploader.value))

({'name': 'Screenshot 2025-10-07 191505.png', 'type': 'image/png', 'size': 100810, 'content': <memory at 0x000002C3AC7BCF40>, 'last_modified': datetime.datetime(2025, 10, 7, 13, 45, 5, 501000, tzinfo=datetime.timezone.utc)},)
<class 'tuple'>


In [15]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [10]:
from langchain.messages import HumanMessage
import sys
sys.path.insert(0,'..')
from ollama_setup import llm
from langchain.agents import create_agent

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])
agent = create_agent(model=llm)
response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

I don't see an image. Can you please describe the image to me? What is it, and what do you want to know about it? I'll do my best to help!


In [17]:
from langchain.messages import HumanMessage
import sys
sys.path.insert(0,'..')
from ollama_setup import llm
from langchain.agents import create_agent

multimodal_question = HumanMessage(content=[
    {"type": "text",
     "text": """Tell me about this image, that i am adding along with this message in data:image/png;base64 format, another question are you equipped to analyze or process image data passed through base64 version string of image  `here is the code i have used      import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")
from langchain.messages import HumanMessage
import sys
sys.path.insert(0,'..')
from ollama_setup import llm
from langchain.agents import create_agent
multimodal_question = HumanMessage(content=[
    {"type": "image_url", "image_url": { "url": f"data:image/png;base64,{img_b64}"}}
      """
    },
    {"type": "image_url", "image_url": { "url": f"data:image/png;base64,{img_b64}"}}
])
agent = create_agent(model=llm)
response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

You're trying to analyze an image that's been base64 encoded and embedded in a message. That's quite an interesting challenge!

Yes, I can help you with that. To analyze the image, we'll need to follow these steps:

1. Decode the base64 string
2. Convert the decoded bytes into a format that can be easily processed by OpenCV or other computer vision libraries.

Here's how you can do it in Python:

```python
import base64
from PIL import Image
import io

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = content_mv.tobytes()

# Decode base64 string
img_b64 = base64.b64decode(img_bytes)

# Save the decoded image to a file-like object (e.g., bytes buffer)
buf = io.BytesIO()
buf.write(img_b64)
buf.seek(0)

# Load the image from the bytes buffer using PIL
img = Image.open(buf)

# Now you can analyze the image using OpenCV or other computer vision libraries.
``

## Audio input

### Step-by-Step Breakdown

### 1. Preparing the Tool

```python
import base64

```

This imports Python’s built-in `base64` module, which contains the functions needed to translate binary data (like images) into readable ASCII text characters.

### 2. Grabbing the Uploaded File

```python
uploaded_file = uploader.value[0]

```

When you use a file uploader widget, it usually stores uploaded files in a list or dictionary. This line accesses `uploader.value` and uses `[0]` to grab the **first (and only) file** from that collection. The result, `uploaded_file`, is a dictionary containing metadata about the file (like its name, size, and content).

### 3. Extracting the Memory View

```python
content_mv = uploaded_file["content"]

```

This extracts the actual file data using the key `"content"`. In many modern Python UI libraries, this data is stored as a `memoryview`.

> **What is a memoryview?** It's a built-in Python object that allows the program to reference the large binary data in your computer's memory directly without making an expensive, slow copy of it first.

### 4. Converting to Bytes

```python
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

```

Because the `base64` encoder cannot work directly with a `memoryview` object, you have to convert it into a standard Python `bytes` object. This line casts the memory view into raw bytes (`img_bytes`).

### 5. Encoding to Base64 Text

```python
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

```

This final line actually performs two operations at once:

* **`base64.b64encode(img_bytes)`**: This converts the raw binary data into Base64 binary data. However, the output of this function is still technically a `bytes` object (e.g., `b'iVBORw0KG...'`).
* **`.decode("utf-8")`**: This turns those Base64 bytes into a standard, clean Python **string** (text).

---

### The Final Result

By the end of this script, `img_b64` will hold a long string of letters, numbers, and symbols that represents your image. You can now easily save this string to a database, send it in a JSON payload to a web server, or use it in an HTML tag like this:

```html
<img src="data:image/png;base64,YOUR_BASE64_STRING_HERE" />

```

Yes, you decode to make the data readable, but it is specifically about converting raw byte markers into standard text characters.Why .decode("utf-8") is NeededType Conversion: b64encode() outputs a bytes object (e.g., b'aW1n...'). .decode("utf-8") converts those bytes into a standard Python string ('aW1n...').JSON Compatibility: You cannot put raw bytes into a JSON payload. You must convert it to a string first.Web Transmission: APIs and web browsers expect text strings when reading JSON or HTML data.Bytes vs. Stringsbase64.b64encode(): This turns your image into a safe ASCII-only sequence, but Python still treats it as raw binary data (bytes)..decode("utf-8"): This tells Python to interpret those ASCII bytes as regular text characters.Code Breakdownpython# 1. Output is binary bytes: b'iVBORw0KGgo...'
raw_b64_bytes = base64.b64encode(img_bytes)

# 2. Output is a clean text string: "iVBORw0KGgo..."
img_b64 = raw_b64_bytes.decode("utf-8")
Use code with caution.To advance your project, let me know:

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.77it/s]

Done.


In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

In [2]:
for i in tqdm(range(50)):
  time.sleep(0.5)

NameError: name 'tqdm' is not defined

In [6]:
tqdm(range(50))

  0%|          | 0/50 [00:00<?, ?it/s]


({'name': 'Screenshot 2025-10-07 191505.png', 'type': 'image/png', 'size': 100810, 'content': <memory at 0x000002C3AC7BCF40>, 'last_modified': datetime.datetime(2025, 10, 7, 13, 45, 5, 501000, tzinfo=datetime.timezone.utc)},)
<class 'tuple'>
Recording...
100%|██████████| 50/50 [00:05<00:00,  9.77it/s]
Done.

I don't see an image. Can you please describe the image to me? What is it, and what do you want to know about it? I'll do my best to help!
You're trying to analyze an image that's been base64 encoded and embedded in a message. That's quite an interesting challenge!

Yes, I can help you with that. To analyze the image, we'll need to follow these steps:

1. Decode the base64 string
2. Convert the decoded bytes into a format that can be easily processed by OpenCV or other computer vision libraries.

Here's how you can do it in Python:

```python
import base64
from PIL import Image
import io

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = content_mv.tobytes()

# Decode base64 string
img_b64 = base64.b64decode(img_bytes)

# Save the decoded image to a file-like object (e.g., bytes buffer)
buf = io.BytesIO()
buf.write(img_b64)
buf.seek(0)

# Load the image from the bytes buffer using PIL
img = Image.open(buf)

# Now you can analyze the image using OpenCV or other computer vision libraries.
```

You'll need to install the `pillow` library, which is a fork of `PIL`, if it's not already installed:

```bash
pip install Pillow
```

With this approach, we've successfully decoded and loaded your base64 encoded image.

As for processing the image data itself, OpenCV or other libraries would be used to perform further analysis (e.g., object detection, image segmentation, etc.). 

Let me know if you need help with any of those steps!

In [ ]:
from tqdm.notebook import tqdm
import time
for _ in tqdm(range(30)):
  time.sleep(0.1)

  0%|          | 0/30 [00:00<?, ?it/s]

In [3]:
with tqdm(desc="Streaming data", unit="packets") as pbar:
    for i in range(10):
        time.sleep(0.5)

        pbar.update(1) # Keeps counting up indefinitely

Streaming data: 10packets [00:05,  1.99packets/s]


In [2]:
from tqdm import tqdm
import time

items = ["apple", "banana", "cherry", "date"]

with tqdm(total=len(items), desc="Processing fruits") as pbar:
    for fruit in items:
        time.sleep(1) # Simulate work

        # Update metrics on the right side of the bar
        pbar.set_postfix(current_item=fruit, status="Done")

        pbar.update(1)


Processing fruits: 100%|██████████| 4/4 [00:04<00:00,  1.00s/it, current_item=date, status=Done]  
